# UTS Graf Sains Data

Nama: Alfathrindra Agastyo Yulianto
NIM: 23031554094
Kelas: 2023 B

Link Github:https://github.com/TyoUnesa/23031554094_Alfathrindra-Agastyo-Yulianto_UTS_Graf_Sains_Data

## 1. Periode training dan testing

| Fold | Periode inklusif | Batas timestamp |
|---|---|---|
| Training (P1) | 17 Mei–17 Juli 2015 | `2015-05-17` sampai sebelum `2015-07-18` UTC |
| Testing (P2) | 26 Juli–10 Agustus 2015 | `2015-07-26` sampai sebelum `2015-08-11` UTC |

Timestamp RetailRocket merupakan Unix epoch dalam milidetik. File mentah tetap disimpan di `data/raw/events.csv`; hasil filter berada di `data/processed/`.

## 2. Penerapan Filter Timestamp dan Penyimpanan CSV

Pada tahap ini, data interaksi RetailRocket difilter berdasarkan periode training (P1) dan testing (P2) yang telah ditentukan. Kolom `timestamp` dikonversi dari Unix epoch dalam milidetik menjadi datetime UTC. Data dari kedua periode disimpan secara terpisah dalam file CSV pada folder `data/processed/`.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display, Image
from scipy.sparse import load_npz

PROJECT = Path.cwd()
RAW = PROJECT / "data" / "raw" / "events.csv"
TRAIN = PROJECT / "data" / "processed" / "events_training.csv"
TEST = PROJECT / "data" / "processed" / "events_testing.csv"
OUT = PROJECT / "outputs"

for p in [RAW, TRAIN, TEST]:
    print(f"{p}: {'ADA' if p.exists() else 'TIDAK DITEMUKAN'}")

C:\Semester 7\Graf Sains Data\UTS\data\raw\events.csv: ADA
C:\Semester 7\Graf Sains Data\UTS\data\processed\events_training.csv: ADA
C:\Semester 7\Graf Sains Data\UTS\data\processed\events_testing.csv: ADA


In [2]:
# %load filter_events.py
import os
import sys

import pandas as pd

SOURCE = os.path.join("data", "raw", "events.csv")
OUTPUT_DIR = os.path.join("data", "processed")

TRAIN_OUTPUT = os.path.join(OUTPUT_DIR, "events_training.csv")
TEST_OUTPUT = os.path.join(OUTPUT_DIR, "events_testing.csv")

CHUNK_SIZE = 100_000

TRAIN_START = pd.Timestamp("2015-05-17", tz="UTC")
TRAIN_END = pd.Timestamp("2015-07-18", tz="UTC")

TEST_START = pd.Timestamp("2015-07-26", tz="UTC")
TEST_END = pd.Timestamp("2015-08-11", tz="UTC")


def main():
    if not os.path.isfile(SOURCE):
        print(f"Error: File '{SOURCE}' tidak ditemukan!")
        sys.exit(1)

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    columns = pd.read_csv(SOURCE, nrows=0).columns

    for output in (TRAIN_OUTPUT, TEST_OUTPUT):
        pd.DataFrame(columns=columns).to_csv(output, index=False)

    training_count = 0
    testing_count = 0
    invalid_timestamps = 0

    print(f"Membaca dataset: {SOURCE}")
    print("Memulai filter timestamp...")

    for chunk in pd.read_csv(SOURCE, chunksize=CHUNK_SIZE):
        timestamp_numeric = pd.to_numeric(
            chunk["timestamp"], errors="coerce"
        )

        dates = pd.to_datetime(
            timestamp_numeric,
            unit="ms",
            utc=True,
            errors="coerce"
        )

        invalid_timestamps += int(dates.isna().sum())

        train_mask = (
            (dates >= TRAIN_START) &
            (dates < TRAIN_END)
        )

        test_mask = (
            (dates >= TEST_START) &
            (dates < TEST_END)
        )

        training = chunk.loc[train_mask]
        testing = chunk.loc[test_mask]

        if not training.empty:
            training.to_csv(
                TRAIN_OUTPUT,
                mode="a",
                header=False,
                index=False
            )

        if not testing.empty:
            testing.to_csv(
                TEST_OUTPUT,
                mode="a",
                header=False,
                index=False
            )

        training_count += len(training)
        testing_count += len(testing)

    print("\nFilter selesai.")
    print(f"Jumlah baris training : {training_count:,}")
    print(f"Jumlah baris testing  : {testing_count:,}")
    print(f"Timestamp tidak valid : {invalid_timestamps:,}")
    print(f"File training: {TRAIN_OUTPUT}")
    print(f"File testing : {TEST_OUTPUT}")


if __name__ == "__main__":
    main()

Membaca dataset: data\raw\events.csv
Memulai filter timestamp...

Filter selesai.
Jumlah baris training : 1,301,201
Jumlah baris testing  : 324,023
Timestamp tidak valid : 0
File training: data\processed\events_training.csv
File testing : data\processed\events_testing.csv


### Hasil filter yang tercatat

- Baris training: **1.301.201**
- Baris testing: **324.023**
- Timestamp tidak valid: **0**

Sel berikut membaca jumlah baris aktual dari CSV (bila file tersedia).

In [3]:
for label, path in [("Training", TRAIN), ("Testing", TEST)]:
    if path.exists():
        with path.open("r", encoding="utf-8") as f:
            n = sum(1 for _ in f) - 1
        print(f"{label}: {n:,} baris")
    else:
        print(f"{label}: file belum ditemukan")

Training: 1,301,201 baris
Testing: 324,023 baris


## 3. Komposisi graf Neo4j: Data Training
Node: `Visitor`, `Item`. Relationship interaksi: `VIEWED`, `ADDED_TO_CART`, `BOUGHT`.

Jalankan query ini di Neo4j Desktop → Query untuk memperoleh komposisi:
```cypher
MATCH (n:Visitor) RETURN 'Visitor' AS component, count(n) AS total
UNION ALL
MATCH (n:Item) RETURN 'Item' AS component, count(n) AS total
UNION ALL
MATCH ()-[r:VIEWED]->() RETURN 'VIEWED' AS component, count(r) AS total
UNION ALL
MATCH ()-[r:ADDED_TO_CART]->() RETURN 'ADDED_TO_CART' AS component, count(r) AS total
UNION ALL
MATCH ()-[r:BOUGHT]->() RETURN 'BOUGHT' AS component, count(r) AS total
ORDER BY component;
```

### Penjelasan query Cypher

Query menggunakan `MATCH` untuk mencari node atau relationship berdasarkan label dan tipe yang terdapat pada graf Neo4j. Pola `MATCH (n:Visitor)` mengambil seluruh node berlabel `Visitor`, sedangkan `MATCH (n:Item)` mengambil seluruh node berlabel `Item`. Fungsi `count(n)` menghitung jumlah node yang ditemukan.

Untuk relationship, pola `MATCH ()-[r:VIEWED]->()` menghitung seluruh relationship bertipe `VIEWED` yang mengarah dari satu node ke node lain. Pola serupa digunakan untuk menghitung relationship `ADDED_TO_CART` dan `BOUGHT`. Fungsi `count(r)` menghitung jumlah relationship yang cocok dengan setiap pola.

`UNION ALL` menggabungkan hasil dari lima query menjadi satu tabel dan mempertahankan setiap baris hasil. Sementara itu, `ORDER BY component` mengurutkan hasil berdasarkan nama komponen graf.

### Verifikasi menggunakan Python API

Query yang sama dijalankan melalui Neo4j Python Driver menggunakan `driver.execute_query()`. Hasil query diterima sebagai kumpulan record, kemudian dikonversi menjadi DataFrame Pandas untuk menampilkan jenis komponen beserta jumlahnya. Hasil Python API sesuai dengan hasil query Cypher yang dijalankan di Neo4j Desktop.

### Hasil yang diperoleh

| Komponen | Jumlah |
|---|---:|
| Visitor | 668.072 |
| Item | 170.631 |
| VIEWED | 1.259.024 |
| ADDED_TO_CART | 31.691 |
| BOUGHT | 10.486 |
| **Total node** | **838.703** |
| **Total relationship interaksi** | **1.301.201** |

Jumlah baris interaksi berbeda dari jumlah interaksi unik di matriks VSM karena pasangan Visitor–Item dapat muncul berulang. Graf mempertahankan kejadian interaksi, sedangkan matriks VSM diubah menjadi biner per pasangan Visitor–Item.

In [ ]:
from getpass import getpass
from neo4j import GraphDatabase

URI = "neo4j://127.0.0.1:7687"
DATABASE = "neo4j"
USERNAME = "neo4j"

composition_query = '''
MATCH (n:Visitor) RETURN 'Visitor' AS component, count(n) AS total
UNION ALL
MATCH (n:Item) RETURN 'Item' AS component, count(n) AS total
UNION ALL
MATCH ()-[r:VIEWED]->() RETURN 'VIEWED' AS component, count(r) AS total
UNION ALL
MATCH ()-[r:ADDED_TO_CART]->() RETURN 'ADDED_TO_CART' AS component, count(r) AS total
UNION ALL
MATCH ()-[r:BOUGHT]->() RETURN 'BOUGHT' AS component, count(r) AS total
ORDER BY component
'''
password = getpass("Password Neo4j (Enter untuk melewati): ")
if password:
    with GraphDatabase.driver(URI, auth=(USERNAME, password)) as driver:
        driver.verify_connectivity()
        records, _, _ = driver.execute_query(composition_query, database_=DATABASE)
    display(pd.DataFrame([dict(r) for r in records]))
else:
    print("Verifikasi live dilewati; hasil terdokumentasi pada tabel di atas.")

## 4. VSM User/Visitor

Setiap baris matriks merepresentasikan Visitor dan setiap kolom Item. Nilai biner `1` menunjukkan pasangan Visitor–Item memiliki setidaknya satu interaksi pada data training. Matriks disimpan dalam format **CSR sparse matrix** karena sangat jarang terisi.

**Hasil tercatat:** 668.072 Visitor, 170.631 Item, shape `(668072, 170631)`, 1.009.792 interaksi unik/non-zero, 113.992.783.640 elemen nol, densitas **0,00088583%**. Plot sampel: 100 Visitor paling aktif dan 238 edge.

In [ ]:
# %load vsm_users.py
import os

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix, save_npz
from sklearn.metrics.pairwise import cosine_similarity


SOURCE = os.path.join(
    "data", "processed", "events_training.csv"
)
OUTPUT_DIR = "outputs"

MATRIX_PATH = os.path.join(
    OUTPUT_DIR, "user_item_vsm.npz"
)
USER_INDEX_PATH = os.path.join(
    OUTPUT_DIR, "visitor_index.csv"
)
ITEM_INDEX_PATH = os.path.join(
    OUTPUT_DIR, "item_index.csv"
)
PLOT_PATH = os.path.join(
    OUTPUT_DIR, "user_projection.png"
)


def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    if not os.path.isfile(SOURCE):
        raise FileNotFoundError(
            f"Dataset tidak ditemukan: {SOURCE}"
        )

    print("Membaca data training...")
    df = pd.read_csv(
        SOURCE,
        usecols=["visitorid", "itemid"]
    )

    user_ids, user_rows = np.unique(
        df["visitorid"].to_numpy(),
        return_inverse=True
    )

    item_ids, item_cols = np.unique(
        df["itemid"].to_numpy(),
        return_inverse=True
    )

    X = coo_matrix(
        (
            np.ones(len(df), dtype=np.float32),
            (user_rows, item_cols)
        ),
        shape=(len(user_ids), len(item_ids)),
        dtype=np.float32
    ).tocsr()

    X.sum_duplicates()
    X.data[:] = 1.0
    X.eliminate_zeros()

    save_npz(MATRIX_PATH, X, compressed=True)

    pd.DataFrame({
        "row_index": np.arange(len(user_ids)),
        "visitorid": user_ids
    }).to_csv(USER_INDEX_PATH, index=False)

    pd.DataFrame({
        "column_index": np.arange(len(item_ids)),
        "itemid": item_ids
    }).to_csv(ITEM_INDEX_PATH, index=False)

    total_possible = X.shape[0] * X.shape[1]
    density = X.nnz / total_possible

    print("\nKARAKTERISTIK VSM USER")
    print("-" * 45)
    print(f"Jumlah User       : {X.shape[0]:,}")
    print(f"Jumlah Item       : {X.shape[1]:,}")
    print(f"Shape matriks     : {X.shape}")
    print(f"Interaksi unik    : {X.nnz:,}")
    print(f"Jumlah elemen nol : {total_possible - X.nnz:,}")
    print(f"Densitas matriks  : {density:.8%}")
    print(f"Format matriks    : {X.getformat()}")
    print(f"Matriks disimpan  : {MATRIX_PATH}")

    sample_size = min(100, X.shape[0])
    activity = np.asarray(X.getnnz(axis=1)).ravel()
    sample_rows = np.argsort(activity)[-sample_size:]

    X_sample = X[sample_rows]
    similarities = cosine_similarity(X_sample)

    G = nx.Graph()

    for idx in sample_rows:
        G.add_node(int(user_ids[idx]))

    k = min(3, max(0, sample_size - 1))

    for i in range(sample_size):
        candidates = np.argsort(similarities[i])[::-1]

        added = 0
        for j in candidates:
            if i == j:
                continue

            score = float(similarities[i, j])

            if score <= 0:
                break

            G.add_edge(
                int(user_ids[sample_rows[i]]),
                int(user_ids[sample_rows[j]]),
                weight=score
            )

            added += 1
            if added >= k:
                break

    plt.figure(figsize=(12, 9))
    pos = nx.spring_layout(G, seed=42, weight="weight")

    nx.draw_networkx(
        G,
        pos=pos,
        with_labels=False,
        node_size=35,
        width=0.7,
        alpha=0.8
    )

    plt.title(
        "User-User Projection (Top 100 Active Users)"
    )
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(PLOT_PATH, dpi=200)
    plt.close()

    print(f"\nNode sampel pada graf : {G.number_of_nodes()}")
    print(f"Edge proyeksi sampel  : {G.number_of_edges()}")
    print(f"Plot disimpan        : {PLOT_PATH}")


if __name__ == "__main__":
    main()

In [ ]:
user_matrix_path = OUT / "user_item_vsm.npz"
user_plot_path = OUT / "user_projection.png"
visitor_index_path = OUT / "visitor_index.csv"
item_index_path = OUT / "item_index.csv"

if user_matrix_path.exists():
    X_user_item = load_npz(user_matrix_path).tocsr()
    total = X_user_item.shape[0] * X_user_item.shape[1]
    print("Shape:", X_user_item.shape)
    print(f"Interaksi unik (nnz): {X_user_item.nnz:,}")
    print(f"Elemen nol: {total - X_user_item.nnz:,}")
    print(f"Densitas: {X_user_item.nnz / total:.8%}")
else:
    print("Matriks tidak ditemukan. Jalankan vsm_users.py terlebih dahulu.")
if user_plot_path.exists():
    display(Image(filename=str(user_plot_path)))
else:
    print("Plot belum ditemukan:", user_plot_path)

**Interpretasi:** Densitas yang sangat rendah menunjukkan bahwa hampir seluruh pasangan Visitor–Item tidak memiliki interaksi. Sparse matrix mengurangi kebutuhan memori. Plot merupakan sampel, bukan keseluruhan graf.

## 5. VSM Item

Matriks Item–User merupakan transpose matriks User–Item: setiap baris adalah Item dan setiap kolom adalah Visitor. Karena orientasi saja yang berubah, jumlah non-zero dan densitasnya tetap sama.

**Hasil tercatat:** 170.631 Item, 668.072 Visitor, shape `(170631, 668072)`, 1.009.792 interaksi unik, densitas **0,00088583%**. Plot sampel: 100 Item paling aktif dan 200 edge.

In [ ]:
# %load vsm_items.py
import os

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from scipy.sparse import load_npz, save_npz
from sklearn.metrics.pairwise import cosine_similarity


OUTPUT_DIR = "outputs"

USER_ITEM_PATH = os.path.join(
    OUTPUT_DIR, "user_item_vsm.npz"
)
ITEM_INDEX_PATH = os.path.join(
    OUTPUT_DIR, "item_index.csv"
)
ITEM_USER_PATH = os.path.join(
    OUTPUT_DIR, "item_user_vsm.npz"
)
PLOT_PATH = os.path.join(
    OUTPUT_DIR, "item_projection.png"
)


def main():
    if not os.path.isfile(USER_ITEM_PATH):
        raise FileNotFoundError(
            f"Matriks User-Item tidak ditemukan: {USER_ITEM_PATH}"
        )

    if not os.path.isfile(ITEM_INDEX_PATH):
        raise FileNotFoundError(
            f"Indeks Item tidak ditemukan: {ITEM_INDEX_PATH}"
        )

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print("Memuat matriks User-Item...")
    X_user_item = load_npz(USER_ITEM_PATH)

    X_item_user = X_user_item.transpose().tocsr()

    # Simpan matriks sparse Item-User.
    save_npz(ITEM_USER_PATH, X_item_user, compressed=True)

    item_ids = pd.read_csv(
        ITEM_INDEX_PATH
    )["itemid"].to_numpy()

    total_possible = (
        X_item_user.shape[0] * X_item_user.shape[1]
    )
    density = X_item_user.nnz / total_possible

    print("\nKARAKTERISTIK VSM ITEM")
    print("-" * 45)
    print(f"Jumlah Item       : {X_item_user.shape[0]:,}")
    print(f"Jumlah User       : {X_item_user.shape[1]:,}")
    print(f"Shape matriks     : {X_item_user.shape}")
    print(f"Interaksi unik    : {X_item_user.nnz:,}")
    print(f"Jumlah elemen nol : {total_possible - X_item_user.nnz:,}")
    print(f"Densitas matriks  : {density:.8%}")
    print(f"Format matriks    : {X_item_user.getformat()}")
    print(f"Matriks disimpan  : {ITEM_USER_PATH}")

    sample_size = min(100, X_item_user.shape[0])
    activity = np.asarray(
        X_item_user.getnnz(axis=1)
    ).ravel()

    sample_rows = np.argsort(activity)[-sample_size:]
    X_sample = X_item_user[sample_rows]

    # Hitung kemiripan antar-Item pada sampel.
    similarities = cosine_similarity(X_sample)

    G = nx.Graph()

    for idx in sample_rows:
        G.add_node(int(item_ids[idx]))

    k = min(3, max(0, sample_size - 1))

    for i in range(sample_size):
        candidates = np.argsort(similarities[i])[::-1]
        added = 0

        for j in candidates:
            if i == j:
                continue

            score = float(similarities[i, j])

            if score <= 0:
                break

            G.add_edge(
                int(item_ids[sample_rows[i]]),
                int(item_ids[sample_rows[j]]),
                weight=score
            )

            added += 1
            if added >= k:
                break

    plt.figure(figsize=(12, 9))

    pos = nx.spring_layout(
        G, seed=42, weight="weight"
    )

    nx.draw_networkx(
        G,
        pos=pos,
        with_labels=False,
        node_size=35,
        width=0.7,
        alpha=0.8
    )

    plt.title(
        "Item-Item Projection (Top 100 Active Items)"
    )
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(PLOT_PATH, dpi=200)
    plt.close()

    print(f"\nNode sampel pada graf : {G.number_of_nodes()}")
    print(f"Edge proyeksi sampel  : {G.number_of_edges()}")
    print(f"Plot disimpan        : {PLOT_PATH}")


if __name__ == "__main__":
    main()


In [ ]:
item_matrix_path = OUT / "item_user_vsm.npz"
item_plot_path = OUT / "item_projection.png"

if item_matrix_path.exists():
    X_item_user = load_npz(item_matrix_path).tocsr()
    total = X_item_user.shape[0] * X_item_user.shape[1]
    print("Shape:", X_item_user.shape)
    print(f"Interaksi unik (nnz): {X_item_user.nnz:,}")
    print(f"Elemen nol: {total - X_item_user.nnz:,}")
    print(f"Densitas: {X_item_user.nnz / total:.8%}")
else:
    print("Matriks tidak ditemukan. Jalankan vsm_items.py terlebih dahulu.")
if item_plot_path.exists():
    display(Image(filename=str(item_plot_path)))
else:
    print("Plot belum ditemukan:", item_plot_path)

## 6. Perbandingan proyeksi


### Benchmark komparatif User–User dan Item–Item

Benchmark ini membandingkan ukuran matriks sparse dan waktu pembentukan proyeksi jaringan pada sampel 100 entitas paling aktif. Kedua proyeksi menggunakan algoritma cosine similarity dan pemilihan hingga tiga tetangga dengan similarity positif. Pengukuran waktu dilakukan berulang dan diringkas menggunakan median untuk mengurangi pengaruh variasi waktu eksekusi.

Perlu diperhatikan bahwa benchmark ini mengukur proyeksi sampel, bukan pembentukan graf penuh. Hasilnya digunakan untuk membandingkan karakteristik komputasi kedua pendekatan, sedangkan pemilihan proyeksi juga mempertimbangkan tujuan analisis.


In [ ]:
from pathlib import Path
from time import perf_counter

import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.sparse import load_npz
from sklearn.metrics.pairwise import cosine_similarity

OUT = Path("outputs")

USER_MATRIX_PATH = OUT / "user_item_vsm.npz"
ITEM_MATRIX_PATH = OUT / "item_user_vsm.npz"
VISITOR_INDEX_PATH = OUT / "visitor_index.csv"
ITEM_INDEX_PATH = OUT / "item_index.csv"

SAMPLE_SIZE = 100
K_NEIGHBORS = 3
REPEATS = 5

required_files = [
    USER_MATRIX_PATH,
    ITEM_MATRIX_PATH,
    VISITOR_INDEX_PATH,
    ITEM_INDEX_PATH,
]

for path in required_files:
    if not path.exists():
        raise FileNotFoundError(
            f"File benchmark tidak ditemukan: {path}"
        )

X_user_item = load_npz(USER_MATRIX_PATH).tocsr()
X_item_user = load_npz(ITEM_MATRIX_PATH).tocsr()

visitor_index = (
    pd.read_csv(VISITOR_INDEX_PATH)
    .sort_values("row_index")
    .reset_index(drop=True)
)

item_index = (
    pd.read_csv(ITEM_INDEX_PATH)
    .sort_values("column_index")
    .reset_index(drop=True)
)

visitor_ids = visitor_index["visitorid"].to_numpy()
item_ids = item_index["itemid"].to_numpy()

if X_user_item.shape != (len(visitor_ids), len(item_ids)):
    raise ValueError(
        "Shape User-Item tidak cocok dengan indeks Visitor dan Item."
    )

if X_item_user.shape != (len(item_ids), len(visitor_ids)):
    raise ValueError(
        "Shape Item-User tidak cocok dengan indeks Item dan Visitor."
    )

difference = X_item_user - X_user_item.transpose()
difference.eliminate_zeros()

if difference.nnz != 0:
    raise ValueError(
        "Matriks Item-User tidak identik dengan transpose User-Item."
    )

print("Validasi orientasi matriks: BERHASIL")
print("X_user_item:", X_user_item.shape)
print("X_item_user:", X_item_user.shape)


def benchmark_projection(matrix, entity_ids, projection_name):
    """Ukur proyeksi k-NN pada sampel entitas paling aktif."""

    t0 = perf_counter()

    activity = np.asarray(matrix.getnnz(axis=1)).ravel()
    sample_size = min(SAMPLE_SIZE, matrix.shape[0])
    sample_rows = np.argsort(activity)[-sample_size:]
    X_sample = matrix[sample_rows]

    sampling_ms = (perf_counter() - t0) * 1000

    cosine_times = []
    graph_times = []
    layout_times = []

    G = None

    for _ in range(REPEATS):

        # A. Hitung cosine similarity.
        start = perf_counter()
        similarities = cosine_similarity(X_sample)
        cosine_times.append(
            (perf_counter() - start) * 1000
        )

        start = perf_counter()

        current_graph = nx.Graph()

        for row in sample_rows:
            current_graph.add_node(
                int(entity_ids[row])
            )

        k = min(K_NEIGHBORS, max(0, sample_size - 1))

        for i in range(sample_size):
            candidates = np.argsort(
                similarities[i]
            )[::-1]

            added = 0

            for j in candidates:
                if i == j:
                    continue

                score = float(similarities[i, j])

                if score <= 0:
                    break

                current_graph.add_edge(
                    int(entity_ids[sample_rows[i]]),
                    int(entity_ids[sample_rows[j]]),
                    weight=score,
                )

                added += 1

                if added >= k:
                    break

        graph_times.append(
            (perf_counter() - start) * 1000
        )

        start = perf_counter()

        nx.spring_layout(
            current_graph,
            seed=42,
            weight="weight",
        )

        layout_times.append(
            (perf_counter() - start) * 1000
        )

        G = current_graph

    matrix_bytes = (
        matrix.data.nbytes
        + matrix.indices.nbytes
        + matrix.indptr.nbytes
    )

    possible_entries = matrix.shape[0] * matrix.shape[1]
    possible_pairs = (
        matrix.shape[0] * (matrix.shape[0] - 1) // 2
    )

    sample_pairs = (
        G.number_of_nodes() * (G.number_of_nodes() - 1) // 2
    )

    result = {
        "Proyeksi": projection_name,
        "Jumlah entitas": matrix.shape[0],
        "Shape": str(matrix.shape),
        "Interaksi unik": matrix.nnz,
        "Densitas matriks (%)": (
            matrix.nnz / possible_entries * 100
        ),
        "Memori CSR (MiB)": matrix_bytes / (1024 ** 2),
        "Pasangan kandidat penuh": possible_pairs,
        "Node sampel": G.number_of_nodes(),
        "Edge sampel": G.number_of_edges(),
        "Densitas edge sampel (%)": (
            G.number_of_edges() / sample_pairs * 100
            if sample_pairs else 0.0
        ),
        "Rata-rata degree sampel": (
            2 * G.number_of_edges() / G.number_of_nodes()
            if G.number_of_nodes() else 0.0
        ),
        "Waktu pemilihan sampel (ms)": sampling_ms,
        "Median cosine (ms)": np.median(cosine_times),
        "Median pembentukan edge (ms)": np.median(graph_times),
        "Median layout (ms)": np.median(layout_times),
        "Pengulangan waktu": REPEATS,
    }

    return result

results = [
    benchmark_projection(
        X_user_item,
        visitor_ids,
        "User-User",
    ),
    benchmark_projection(
        X_item_user,
        item_ids,
        "Item-Item",
    ),
]

benchmark_df = pd.DataFrame(results)

display(
    benchmark_df.style.format({
        "Densitas matriks (%)": "{:.8f}",
        "Memori CSR (MiB)": "{:.3f}",
        "Densitas edge sampel (%)": "{:.3f}",
        "Rata-rata degree sampel": "{:.3f}",
        "Waktu pemilihan sampel (ms)": "{:.3f}",
        "Median cosine (ms)": "{:.3f}",
        "Median pembentukan edge (ms)": "{:.3f}",
        "Median layout (ms)": "{:.3f}",
    })
)

benchmark_path = OUT / "projection_benchmark.csv"
benchmark_df.to_csv(benchmark_path, index=False)

print(f"\nHasil benchmark disimpan ke: {benchmark_path}")


### Kesimpulan perbandingan proyeksi

Berdasarkan hasil eksplorasi dan benchmark, proyeksi **Item–Item dipilih sebagai pendekatan yang lebih efisien untuk analisis similarity pada dataset RetailRocket**. Jumlah entitas Item sebanyak 170.631 lebih kecil daripada jumlah Visitor sebanyak 668.072, sehingga ruang kandidat pasangan entitas Item–Item secara teoretis sekitar 15,3 kali lebih kecil. Pengukuran juga menunjukkan bahwa representasi matriks CSR Item–Item menggunakan memori sebesar 8,355 MiB, lebih rendah daripada User–User sebesar 10,253 MiB. Pada benchmark ini, waktu pemilihan sampel, pembentukan edge, dan perhitungan layout NetworkX juga lebih rendah untuk Item–Item, meskipun perhitungan cosine similarity sedikit lebih cepat pada User–User.

Dari sisi efektivitas, proyeksi Item–Item lebih sesuai dengan tujuan analisis berikutnya, yaitu mencari lima Item yang memiliki pola interaksi serupa berdasarkan cosine similarity. Proyeksi User–User tetap berguna apabila tujuan analisisnya adalah menemukan Visitor dengan pola interaksi yang mirip. Jumlah edge pada kedua plot sampel tidak dijadikan satu-satunya dasar pemilihan karena simpul yang divisualisasikan berbeda.

Dengan demikian, **Item–Item dipilih sebagai proyeksi utama untuk perhitungan similarity pada soal nomor 7**. Kesimpulan efisiensi ini berlaku berdasarkan representasi matriks dan benchmark sampel yang diuji; waktu komputasi graf penuh belum diukur secara langsung.

## 7. Perhitungan Cosine Similarity pada Proyeksi Item–Item

Berdasarkan hasil perbandingan proyeksi pada nomor 6, pendekatan Item–Item digunakan untuk menghitung kemiripan antar-Item. Item acuan dipilih secara reproducible berdasarkan jumlah Visitor unik terbanyak, yaitu Item 5411 dengan 1.514 Visitor unik.

Cosine similarity dihitung menggunakan vektor interaksi Item–Visitor. Perhitungan proyeksi menghasilkan jumlah Visitor bersama antar-Item, yang kemudian dinormalisasi menggunakan jumlah Visitor unik masing-masing Item. Lima Item dengan nilai cosine similarity tertinggi terhadap Item acuan disimpan ke dalam CSV untuk diverifikasi dan dituliskan ke Neo4j.

In [ ]:
# %load calculate_similarity.py
import os

import numpy as np
import pandas as pd
from scipy.sparse import load_npz


OUTPUT_DIR = "outputs"

MATRIX_PATH = os.path.join(
    OUTPUT_DIR, "item_user_vsm.npz"
)
ITEM_INDEX_PATH = os.path.join(
    OUTPUT_DIR, "item_index.csv"
)
RESULT_PATH = os.path.join(
    OUTPUT_DIR, "top5_item_similarity.csv"
)


def main():
    print("Memuat matriks Item-User...")

    if not os.path.isfile(MATRIX_PATH):
        raise FileNotFoundError(
            f"Matriks tidak ditemukan: {MATRIX_PATH}"
        )

    if not os.path.isfile(ITEM_INDEX_PATH):
        raise FileNotFoundError(
            f"Indeks Item tidak ditemukan: {ITEM_INDEX_PATH}"
        )

    matrix = load_npz(MATRIX_PATH).tocsr()

    item_index = (
        pd.read_csv(ITEM_INDEX_PATH)
        .sort_values("column_index")
        .reset_index(drop=True)
    )

    if matrix.shape[0] != len(item_index):
        raise ValueError(
            "Jumlah baris matriks tidak sesuai dengan indeks Item."
        )

    item_ids = item_index["itemid"].to_numpy(dtype=np.int64)

    # Derajat Item = jumlah Visitor unik yang berinteraksi dengannya.
    item_degrees = np.asarray(
        matrix.getnnz(axis=1)
    ).ravel()

    # Pemilihan Item acuan reproducible:
    # Item dengan jumlah Visitor unik terbanyak.
    reference_idx = int(np.argmax(item_degrees))
    reference_item_id = int(item_ids[reference_idx])

    print(f"Item acuan       : {reference_item_id}")
    print(f"Jumlah Visitor unik: {item_degrees[reference_idx]:,}")

    # PROYEKSI ITEM-ITEM:
    # P[i,j] menghitung jumlah Visitor bersama antara Item i dan j.
    # Hitung satu baris proyeksi saja agar tidak membangun seluruh
    # matriks Item-Item sekaligus.
    print("Menghitung bobot proyeksi Item-Item...")

    projected_row = (
        matrix[reference_idx] @ matrix.T
    ).toarray().ravel()

    # Cosine similarity dari bobot proyeksi:
    # common_visitors / sqrt(degree_item_i * degree_item_j)
    denominator = np.sqrt(
        item_degrees[reference_idx] * item_degrees
    )

    scores = np.divide(
        projected_row,
        denominator,
        out=np.zeros_like(projected_row, dtype=float),
        where=denominator > 0
    )

    # Item acuan tidak boleh menjadi rekomendasi untuk dirinya sendiri.
    scores[reference_idx] = 0.0

    candidate_indices = np.flatnonzero(scores > 0)

    # Urutkan dari similarity terbesar. Jika nilainya sama,
    # gunakan ID Item untuk menentukan urutan secara konsisten.
    ranked_indices = candidate_indices[
        np.lexsort((
            item_ids[candidate_indices],
            -scores[candidate_indices]
        ))
    ]

    top_indices = ranked_indices[:5]

    if len(top_indices) < 5:
        print(
            f"Peringatan: hanya ditemukan {len(top_indices)} "
            "Item dengan similarity positif."
        )

    results = pd.DataFrame({
        "reference_itemid": reference_item_id,
        "similar_itemid": item_ids[top_indices],
        "similarity": scores[top_indices],
        "similar_item_index": top_indices
    })

    results.to_csv(RESULT_PATH, index=False)

    print("\nTOP 5 ITEM DARI PROYEKSI ITEM-ITEM")
    print("-" * 65)
    print(results.to_string(
        index=False,
        formatters={"similarity": lambda x: f"{x:.8f}"}
    ))
    print("-" * 65)
    print(f"Hasil disimpan ke: {RESULT_PATH}")


if __name__ == "__main__":
    main()

In [ ]:
similarity_csv = OUT / "top5_item_similarity.csv"
if similarity_csv.exists():
    similarity_results = pd.read_csv(similarity_csv)
    display(similarity_results)
else:
    print("Hasil belum ada. Jalankan calculate_similarity.py terlebih dahulu.")

### Lima Item dengan similarity tertinggi terhadap Item 5411

| Item acuan | Item serupa | Cosine similarity |
|---:|---:|---:|
| 5411 | 33030 | 0.15015033 |
| 5411 | 33956 | 0.09086404 |
| 5411 | 358234 | 0.08715370 |
| 5411 | 459936 | 0.063675866 |
| 5411 | 375492 | 0.06356836 |

CSV hasil: `outputs/top5_item_similarity.csv`.

### 7(a). Query Cypher untuk lima similarity tertinggi

Query Cypher berikut digunakan untuk mengambil lima Item yang memiliki nilai cosine similarity tertinggi terhadap Item acuan 5411. Query membaca relationship `SIMILAR_TO` yang telah tersimpan pada graf Neo4j, mengambil nilai properti `similarity`, mengurutkan hasil dari nilai tertinggi ke terendah, dan membatasi hasil menjadi lima record.

Query verifikasi:
```cypher
MATCH (source:Item {itemId: 5411})-[r:SIMILAR_TO]->(target:Item)
RETURN source.itemId AS reference_itemid,
       target.itemId AS similar_itemid,
       r.similarity AS similarity
ORDER BY similarity DESC
LIMIT 5;
```

### 7(b). Penulisan similarity ke graf Neo4j dan verifikasi

Nilai cosine similarity yang telah dihitung menggunakan Python ditulis ke graf Neo4j sebagai relationship berarah `SIMILAR_TO`. Setiap relationship menghubungkan Item acuan dengan Item serupa dan menyimpan nilai numerik pada properti `similarity`. Proses penulisan menggunakan `MERGE` untuk mencocokkan atau membuat relationship berdasarkan pasangan node, kemudian `SET` untuk menetapkan nilai similarity.

In [ ]:
# %load write_similarity_to_neo4j.py
import os
from getpass import getpass

import pandas as pd
from neo4j import GraphDatabase


URI = "neo4j://127.0.0.1:7687"
DATABASE = "neo4j"
USERNAME = "neo4j"

INPUT_PATH = os.path.join(
    "outputs", "top5_item_similarity.csv"
)

QUERY = """
UNWIND $rows AS row

MATCH (source:Item {itemId: row.reference_itemid})
MATCH (target:Item {itemId: row.similar_itemid})

MERGE (source)-[r:SIMILAR_TO]->(target)
SET r.similarity = row.similarity

RETURN count(r) AS relationships_written
"""


def main():
    if not os.path.isfile(INPUT_PATH):
        raise FileNotFoundError(
            f"File hasil similarity tidak ditemukan: {INPUT_PATH}"
        )

    df = pd.read_csv(INPUT_PATH)

    required_columns = {
        "reference_itemid",
        "similar_itemid",
        "similarity"
    }

    if not required_columns.issubset(df.columns):
        raise ValueError(
            f"Kolom CSV harus mencakup: {required_columns}"
        )

    rows = [
        {
            "reference_itemid": int(row.reference_itemid),
            "similar_itemid": int(row.similar_itemid),
            "similarity": float(row.similarity)
        }
        for row in df.itertuples(index=False)
    ]

    password = getpass("Password Neo4j: ")

    with GraphDatabase.driver(
        URI, auth=(USERNAME, password)
    ) as driver:
        driver.verify_connectivity()

        records, _, _ = driver.execute_query(
            QUERY,
            rows=rows,
            database_=DATABASE
        )

        written = records[0]["relationships_written"]

        print(f"Jumlah baris kandidat: {len(rows)}")
        print(f"Relationship diproses: {written}")

        if written != len(rows):
            print(
                "PERINGATAN: Tidak semua pasangan berhasil "
                "diproses. Periksa apakah seluruh Item tersedia "
                "di database training."
            )
        else:
            print(
                "Semua relationship SIMILAR_TO berhasil disimpan."
            )


if __name__ == "__main__":
    main()


#### Verifikasi hasil penulisan dengan query Cypher

Query berikut digunakan untuk memeriksa bahwa kelima relationship `SIMILAR_TO` dari Item acuan 5411 telah tersimpan beserta nilai properti `similarity`. Hasilnya dibandingkan dengan tabel perhitungan cosine similarity dari Python untuk memastikan bahwa nilai yang tersimpan pada graf sesuai dengan hasil perhitungan.

In [ ]:
similarity_query = '''
MATCH (source:Item {itemId: 5411})-[r:SIMILAR_TO]->(target:Item)
RETURN source.itemId AS reference_itemid,
       target.itemId AS similar_itemid,
       r.similarity AS similarity
ORDER BY similarity DESC
LIMIT 5
'''
password = getpass("Password Neo4j (Enter untuk melewati): ")
if password:
    with GraphDatabase.driver(URI, auth=(USERNAME, password)) as driver:
        driver.verify_connectivity()
        records, _, _ = driver.execute_query(similarity_query, database_=DATABASE)
    display(pd.DataFrame([dict(r) for r in records]))
else:
    print("Verifikasi live dilewati; lihat tabel hasil yang tercatat.")

## 8. Visualisasi Graf Sebelum dan Sesudah Similarity

### 8(a). Graf sebelum similarity

Query sampel graf interaksi di Neo4j Desktop:
```cypher
MATCH (v:Visitor)-[r]->(i:Item)
RETURN v, r, i
LIMIT 100;
```
Screenshot yang tercatat menunjukkan 85 node (80 Visitor dan 5 Item) serta 100 relationship interaksi pada sampel.

In [ ]:
from IPython.display import display, Image
from pathlib import Path

before_path = Path("outputs") / "graph_before_similarity.png"

if before_path.exists():
    display(Image(filename=str(before_path)))
else:
    print(f"Gambar tidak ditemukan: {before_path}")

### 8(b). Graf sesudah similarity

Query graf similarity:
```cypher
MATCH (source:Item {itemId: 5411})-[r:SIMILAR_TO]->(target:Item)
RETURN source, r, target;
```
Screenshot yang tercatat menunjukkan 6 node Item (Item acuan dan lima Item serupa) serta 5 relationship `SIMILAR_TO`.

In [ ]:
from IPython.display import display, Image
from pathlib import Path

after_path = Path("outputs") / "graph_after_similarity.png"

if after_path.exists():
    display(Image(filename=str(after_path)))
else:
    print(f"Gambar tidak ditemukan: {after_path}")

## Kesimpulan Bagian 2.1

1. Data interaksi RetailRocket dibagi berdasarkan periode temporal menjadi data training (P1) dan testing (P2), kemudian disimpan dalam dua file CSV terpisah.

2. Graf training Neo4j terdiri atas 668.072 node `Visitor`, 170.631 node `Item`, serta 1.301.201 relationship interaksi yang terdiri atas `VIEWED`, `ADDED_TO_CART`, dan `BOUGHT`.

3. VSM User–Item memiliki shape `(668072, 170631)` dengan 1.009.792 pasangan Visitor–Item unik. Matriks disimpan dalam format CSR sparse matrix karena sebagian besar elemennya bernilai nol.

4. VSM Item–User merupakan transpose dari VSM User–Item dengan shape `(170631, 668072)`. Jumlah elemen non-zero dan densitas kedua matriks tetap sama karena keduanya merepresentasikan interaksi yang sama dengan orientasi berbeda.

5. Proyeksi Item–Item dipilih untuk analisis similarity karena jumlah entitas Item lebih sedikit daripada Visitor dan pendekatannya sesuai dengan tujuan pencarian Item yang memiliki pola interaksi serupa. Benchmark pada sampel juga menunjukkan penggunaan memori CSR Item–Item lebih rendah daripada User–User.

6. Lima Item dengan cosine similarity tertinggi terhadap Item acuan 5411 dihitung menggunakan Python, kemudian nilai similarity disimpan dan diverifikasi pada relationship `SIMILAR_TO` di Neo4j.

7. Visualisasi graf sebelum dan sesudah penambahan relationship `SIMILAR_TO` dihasilkan melalui query Neo4j dan ditampilkan pada notebook. Graf sebelum similarity menunjukkan sampel interaksi Visitor–Item, sedangkan graf sesudah similarity menunjukkan Item acuan 5411 beserta lima Item serupa dan nilai similarity-nya.

## 2.2 Analytics KNN, Hit@5, dan NDCG@5

### 1. Pemilihan User dan rekomendasi KNN

Pemilihan User dilakukan secara acak dan dapat direproduksi menggunakan random seed 42. User yang dipilih adalah Visitor ID 28250, dengan tujuh Item unik pada data training dan satu Item relevan yang belum pernah diinteraksikan pada periode testing.

Metode yang digunakan adalah User-based K-Nearest Neighbors (KNN) dengan \(k=5\). Kemiripan antar-User dihitung menggunakan cosine similarity pada matriks interaksi User–Item dari data training. Lima tetangga terdekat kemudian digunakan untuk membentuk skor rekomendasi Item berdasarkan penjumlahan berbobot interaksi tetangga.

Tiga Item rekomendasi teratas untuk Visitor 28250 adalah:

| Rank | Item ID | Skor |
|---:|---:|---:|
| 1 | 33719 | 0.591608 |
| 2 | 53545 | 0.591608 |
| 3 | 104752 | 0.591608 |

In [ ]:
# %load analytics_knn.py
import os
import numpy as np
import pandas as pd
from scipy.sparse import load_npz
from sklearn.metrics.pairwise import cosine_similarity

OUTPUT_DIR = "outputs"
DATA_DIR = os.path.join("data", "processed")

TRAIN_CSV = os.path.join(DATA_DIR, "events_training.csv")
TEST_CSV = os.path.join(DATA_DIR, "events_testing.csv")

MATRIX_PATH = os.path.join(OUTPUT_DIR, "user_item_vsm.npz")
VISITOR_INDEX_PATH = os.path.join(OUTPUT_DIR, "visitor_index.csv")
ITEM_INDEX_PATH = os.path.join(OUTPUT_DIR, "item_index.csv")

RECOMMENDATIONS_PATH = os.path.join(
    OUTPUT_DIR, "knn_recommendations.csv"
)
NEIGHBORS_PATH = os.path.join(
    OUTPUT_DIR, "knn_neighbors.csv"
)
METRICS_PATH = os.path.join(
    OUTPUT_DIR, "knn_metrics.csv"
)

RANDOM_SEED = 42
K = 5
TOP_N = 5


def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    required_files = [
        TRAIN_CSV,
        TEST_CSV,
        MATRIX_PATH,
        VISITOR_INDEX_PATH,
        ITEM_INDEX_PATH,
    ]

    for path in required_files:
        if not os.path.isfile(path):
            raise FileNotFoundError(f"File tidak ditemukan: {path}")

    print("Memuat matriks User-Item...")
    X = load_npz(MATRIX_PATH).tocsr()

    visitor_index = (
        pd.read_csv(VISITOR_INDEX_PATH)
        .sort_values("row_index")
        .reset_index(drop=True)
    )

    item_index = (
        pd.read_csv(ITEM_INDEX_PATH)
        .sort_values("column_index")
        .reset_index(drop=True)
    )

    visitor_ids = visitor_index["visitorid"].to_numpy(dtype=np.int64)
    item_ids = item_index["itemid"].to_numpy(dtype=np.int64)

    if X.shape != (len(visitor_ids), len(item_ids)):
        raise ValueError("Dimensi matriks tidak cocok dengan indeks.")

    visitor_to_row = {
        int(visitor_id): row
        for row, visitor_id in enumerate(visitor_ids)
    }

    item_to_col = {
        int(item_id): col
        for col, item_id in enumerate(item_ids)
    }

    print("Shape matriks training:", X.shape)

    print("Membaca interaksi periode testing...")

    test_df = pd.read_csv(
        TEST_CSV,
        usecols=["visitorid", "itemid"],
        dtype={"visitorid": "int64", "itemid": "int64"},
    )

    test_df = test_df[
        test_df["visitorid"].isin(visitor_to_row)
        & test_df["itemid"].isin(item_to_col)
    ]

    relevant_by_user = {}

    for visitor_id, group in test_df.groupby("visitorid"):
        visitor_id = int(visitor_id)
        user_row = visitor_to_row[visitor_id]

        seen_columns = set(X.getrow(user_row).indices.tolist())

        test_items = set(group["itemid"].unique().tolist())

        unseen_relevant = {
            int(item_id)
            for item_id in test_items
            if item_to_col[int(item_id)] not in seen_columns
        }

        if unseen_relevant:
            relevant_by_user[visitor_id] = unseen_relevant

    eligible_users = sorted(relevant_by_user.keys())

    if not eligible_users:
        raise ValueError(
            "Tidak ada User yang memiliki Item relevan pada periode testing."
        )

    users_with_history = [
        uid for uid in eligible_users
        if X.getrow(visitor_to_row[uid]).nnz >= TOP_N
    ]

    if not users_with_history:
        raise ValueError(
            "Tidak ditemukan User dengan minimal 5 Item di training "
            "dan Item relevan di testing."
        )

    rng = np.random.default_rng(RANDOM_SEED)
    candidate_order = rng.permutation(
        np.asarray(users_with_history, dtype=np.int64)
    )

    selected_user_id = None
    selected_row = None
    neighbor_rows = None
    neighbor_scores = None
    item_scores = None
    top_columns = None

    print("Mencari User secara acak dengan seed:", RANDOM_SEED)

    for candidate_user in candidate_order:
        candidate_user = int(candidate_user)
        row = visitor_to_row[candidate_user]
        user_vector = X.getrow(row)

        sims = cosine_similarity(user_vector, X).ravel()
        sims[row] = 0.0

        candidate_neighbors = np.flatnonzero(sims > 0)

        if len(candidate_neighbors) < K:
            continue

        order = candidate_neighbors[
            np.lexsort((
                candidate_neighbors,
                -sims[candidate_neighbors],
            ))
        ]

        local_neighbor_rows = order[:K]
        local_neighbor_scores = sims[local_neighbor_rows]

        neighbor_matrix = X[local_neighbor_rows]
        local_item_scores = np.asarray(
            neighbor_matrix.T @ local_neighbor_scores
        ).ravel()

        local_item_scores[user_vector.indices] = 0.0

        candidate_items = np.flatnonzero(local_item_scores > 0)

        if len(candidate_items) < TOP_N:
            continue

        local_top_columns = candidate_items[
            np.lexsort((
                item_ids[candidate_items],
                -local_item_scores[candidate_items],
            ))
        ][:TOP_N]

        selected_user_id = candidate_user
        selected_row = row
        neighbor_rows = local_neighbor_rows
        neighbor_scores = local_neighbor_scores
        item_scores = local_item_scores
        top_columns = local_top_columns
        break

    if selected_user_id is None:
        raise ValueError(
            "Tidak ditemukan User yang memenuhi kriteria setelah "
            "pemeriksaan kandidat. Periksa data training dan testing."
        )

    relevant_items = relevant_by_user[selected_user_id]

    print("\nUSER TERPILIH")
    print("-" * 55)
    print("Random seed                  :", RANDOM_SEED)
    print("Visitor ID                   :", selected_user_id)
    print("Item unik di training        :", X.getrow(selected_row).nnz)
    print("Item relevan baru di testing :", len(relevant_items))

    neighbors_df = pd.DataFrame({
        "neighbor_rank": np.arange(1, K + 1),
        "neighbor_visitorid": visitor_ids[neighbor_rows],
        "cosine_similarity": neighbor_scores,
    })

    neighbors_df.to_csv(NEIGHBORS_PATH, index=False)

    print("\nK-NEAREST NEIGHBORS")
    print(neighbors_df.to_string(
        index=False,
        formatters={
            "cosine_similarity": lambda x: f"{x:.6f}"
        },
    ))

    recommended_item_ids = item_ids[top_columns]

    recommendation_relevance = np.array([
        int(int(item_id) in relevant_items)
        for item_id in recommended_item_ids
    ])

    recommendations_df = pd.DataFrame({
        "visitorid": selected_user_id,
        "rank": np.arange(1, TOP_N + 1),
        "itemid": recommended_item_ids,
        "score": item_scores[top_columns],
        "relevant_in_testing": recommendation_relevance,
    })

    recommendations_df.to_csv(RECOMMENDATIONS_PATH, index=False)

    print("\nTOP 5 REKOMENDASI ITEM")
    print(recommendations_df.to_string(
        index=False,
        formatters={"score": lambda x: f"{x:.6f}"},
    ))

    print("\nTIGA ITEM UNTUK JAWABAN SOAL 2.2.1(b)")
    for row in recommendations_df.head(3).itertuples(index=False):
        print(f"Rank {row.rank}: Item {row.itemid} (skor={row.score:.6f})")

    hit_at_5 = float(np.any(recommendation_relevance == 1))

    discounts = 1.0 / np.log2(np.arange(2, TOP_N + 2))
    dcg_at_5 = float(np.sum(recommendation_relevance * discounts))

    ideal_length = min(len(relevant_items), TOP_N)
    ideal_discounts = 1.0 / np.log2(
        np.arange(2, ideal_length + 2)
    )
    idcg_at_5 = float(np.sum(ideal_discounts))

    ndcg_at_5 = dcg_at_5 / idcg_at_5 if idcg_at_5 > 0 else 0.0

    metrics_df = pd.DataFrame([{
        "random_seed": RANDOM_SEED,
        "visitorid": selected_user_id,
        "k_neighbors": K,
        "recommendation_cutoff": TOP_N,
        "relevant_test_items": len(relevant_items),
        "hits_in_top5": int(recommendation_relevance.sum()),
        "hit_at_5": hit_at_5,
        "dcg_at_5": dcg_at_5,
        "idcg_at_5": idcg_at_5,
        "ndcg_at_5": ndcg_at_5,
    }])

    metrics_df.to_csv(METRICS_PATH, index=False)

    print("\nEVALUASI REKOMENDASI")
    print("-" * 55)
    print(f"Hit@5  : {hit_at_5:.4f}")
    print(f"DCG@5  : {dcg_at_5:.6f}")
    print(f"IDCG@5 : {idcg_at_5:.6f}")
    print(f"NDCG@5 : {ndcg_at_5:.4f}")

    print("\nFile hasil:")
    print("-", NEIGHBORS_PATH)
    print("-", RECOMMENDATIONS_PATH)
    print("-", METRICS_PATH)


if __name__ == "__main__":
    main()

### 2. Evaluasi rekomendasi

Berdasarkan pembandingan hasil rekomendasi dengan Item relevan pada periode testing, seluruh lima Item rekomendasi memiliki label relevansi 0.

**Hit@5** mengukur apakah setidaknya satu Item relevan ditemukan dalam lima rekomendasi teratas. Karena tidak terdapat Item relevan, diperoleh Hit@5 = 0.

**NDCG@5** mengukur kualitas urutan rekomendasi berdasarkan relevansi. Hasil penghitungan memberikan DCG@5 = 0 dan IDCG@5 = 1, sehingga NDCG@5 = 0.

Nilai tersebut menggambarkan hasil untuk satu User yang dipilih dengan seed 42. Hasil ini tidak dapat langsung digeneralisasi menjadi kinerja rata-rata sistem pada seluruh User.
